In [0]:
%sql
SELECT *
FROM read_files(
  'file:/Workspace/Users/kacperjwawrzyniak@gmail.com/IC8/orders_seed.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

df = spark.read.csv(
'file:/Workspace/Users/kacperjwawrzyniak@gmail.com/IC8/orders_seed.csv',
header=True,
inferSchema=True
)

df = df.withColumn('year', F.year('order_date'))

df.printSchema(),
df.show()



In [0]:
%sql
SHOW VOLUMES IN prod.sales;

CREATE VOLUME IF NOT EXISTS prod.sales.raw_files;

SHOW VOLUMES IN prod.sales;

--/Volumes/prod/sales/raw_files/orders_parquet


In [0]:
    
#df.write.mode("overwrite").format("parquet").partitionBy("year").save("/Volumes/prod/sales/raw_files/orders_parquet")


In [0]:
df.write.mode("overwrite").format("parquet").partitionBy("year").saveAsTable("prod.sales.orders_parquet")

#It's not work because Unity Catalog managed table can't be Parquet.

In [0]:
%sql
DESCRIBE DETAIL prod.sales.orders_parquet;

In [0]:
# ## HW8.1 — Convert Parquet → Delta

# First, I read the data from the `orders_seed.csv` file and added the `year` column based on `order_date`.

# Next, I saved the data as Parquet files partitioned by `year` to the path:

# `/Volumes/prod/sales/raw_files/orders_parquet`

# I confirmed the save was successful with the `LIST` command, which showed the `year=2026/` partition directory and the `_SUCCESS` file.

# Attempting to execute `CONVERT TO DELTA` directly on the `/Volumes/...` path resulted in the environment error `INVALID_DBFS_MOUNT_USER_ERROR`.

# Next, I tried to create a managed table in Parquet format using `saveAsTable`, but Unity Catalog returned the error: `Only Delta is supported for managed tables. The provided datasource format is PARQUET.

# Conclusion: The Parquet catalog was prepared correctly, but in this environment, in-place conversion requires a properly configured external location/external table or a different access configuration. As a training workaround, you can perform a Parquet → Delta migration by reading the Parquet catalog into a DataFrame and storing the result as a Delta table.